# 单文件 LFP 流程

本 Notebook 可在真实 FIF 到位时运行真实文件；若路径不存在，则自动切换到明确标记的合成信号，仅用于算法验证。它不补造动物身份、给药信息或 AIMs。

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from lfp_analysis.pipeline import run_single_file
from lfp_analysis.synthetic import validate_synthetic

sample_path = Path(r'C:\Users\PC\Desktop\94\LID-94\T80\LID-T80_all_channels-epo.fif')
real_output = PROJECT_ROOT / 'results' / 'notebook_LID-T80'
if sample_path.exists():
    manifest = run_single_file(sample_path, PROJECT_ROOT / 'configs' / 'default.yaml', real_output, PROJECT_ROOT / 'metadata')
    run_mode = 'REAL FIF (local read-only input)'
else:
    manifest = validate_synthetic(PROJECT_ROOT / 'results' / 'notebook_synthetic_validation')
    run_mode = 'SYNTHETIC VALIDATION ONLY'
print(run_mode)
manifest

REAL FIF (local read-only input)


{'analysis_version': '0.1.0',
 'input_path': 'C:\\Users\\PC\\Desktop\\94\\LID-94\\T80\\LID-T80_all_channels-epo.fif',
 'input_sha256': '2751bce6cdad2a17d7ba72978b0b59221f8265dd27d9c77a8e77600ed75a0582',
 'input_size_bytes': 6722922,
 'file_id': 'LID-T80_all_channels-epo',
 'identity_status': 'file_only_identity_unresolved',
 'registry_row': {'file_id': 'LID-T80_all_channels-epo',
  'session_id': nan,
  'animal_id': nan,
  'file_path': 'C:\\Users\\PC\\Desktop\\94\\LID-94\\T80\\LID-T80_all_channels-epo.fif',
  'dose_state': 'post_dose',
  'drug': 'L-DOPA',
  'nominal_dose_time_min': '80',
  'actual_record_start': nan,
  'actual_record_end': nan,
  'preprocessing_notes': '预处理历史待核实；T80为名义给药后时间；动物身份和session未提供',
  'include_file_level': 'true',
  'is_example': 'known_sample_unresolved_identity'},
 'n_epochs': 21,
 'n_channels': 16,
 'n_times': 5000,
 'sampling_rate_hz': 1000.0,
 'epoch_tmin_s': 0.0,
 'epoch_tmax_s': 4.999,
 'effective_valid_duration_s': 105.0,
 'events_are_retained_raw_value

In [2]:
if 'REAL' in run_mode:
    quality_file = pd.read_csv(real_output / 'quality_file.csv')
    quality_channel = pd.read_csv(real_output / 'quality_channel.csv')
    epochs_trace = pd.read_csv(real_output / 'epochs_trace.csv')
    display(quality_file)
    display(quality_channel.loc[quality_channel['n_warn_epochs'] > 0])
    display(epochs_trace.loc[epochs_trace['drop_reason'].notna() & epochs_trace['drop_reason'].ne('')])
    parameterization_model = pd.read_csv(real_output / 'parameterization_model.csv')
    parameterization_peaks = pd.read_csv(real_output / 'parameterization_peaks.csv')
    display(parameterization_model[['channel_name', 'backend_used', 'fit_status', 'r_squared', 'offset', 'exponent']])
    display(parameterization_peaks.groupby('channel_name', as_index=False).size().rename(columns={'size': 'n_peaks'}))
else:
    display(manifest)

,n_epochs,n_channels,n_times,sampling_rate_hz,nominal_duration_s,effective_valid_duration_s,n_fail_epoch_channel_rows,n_warn_epoch_channel_rows,n_duplicate_epoch_rows
0,21,16,5000,1000.0,105.0,105.0,0,1,0


,channel_array_index,channel_name,n_epochs,n_fail_epochs,n_warn_epochs,mean_std,max_abs,max_issue_score
12,12,TETFP21,21,0,1,0.000112,0.001986,1


,file_id,saved_index,original_candidate_index,events_raw_value,selection_value,confirmed_time_start_s,confirmed_time_end_s,quality_status,quality_flags,drop_reason,time_coordinate_note
4,LID-T80_all_channels-epo,NaN,4,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
6,LID-T80_all_channels-epo,NaN,6,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
8,LID-T80_all_channels-epo,NaN,8,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...
10,LID-T80_all_channels-epo,NaN,10,NaN,NaN,NaN,NaN,dropped_upstream,NaN,USER,events raw value retained; not interpreted as ...


,channel_name,backend_used,fit_status,r_squared,offset,exponent
0,TETFP01,SpectralModel,ok,0.991758,-8.677936,1.470203
1,TETFP02,SpectralModel,ok,0.993069,-8.774754,1.433505
2,TETFP03,SpectralModel,ok,0.993186,-8.661396,1.454898
3,TETFP04,SpectralModel,ok,0.990925,-8.839657,1.379560
4,TETFP05,SpectralModel,ok,0.985538,-8.871392,1.801595
5,TETFP06,SpectralModel,ok,0.980416,-8.825426,1.799210
6,TETFP07,SpectralModel,ok,0.984628,-8.639296,1.894939
7,TETFP08,SpectralModel,ok,0.973294,-9.139116,1.689818
8,TETFP17,SpectralModel,ok,0.995653,-9.174448,1.259565
9,TETFP18,SpectralModel,ok,0.995297,-9.340494,1.206834


,channel_name,n_peaks
0,TETFP01,5
1,TETFP02,6
2,TETFP03,6
3,TETFP04,6
4,TETFP05,6
5,TETFP06,6
6,TETFP07,6
7,TETFP08,6
8,TETFP17,6
9,TETFP18,6


## 解释边界

T80 是名义给药后时点；本流程不把 epoch 的 events 值解释成原始记录起止时间。没有登记动物身份、给药天数和行为同步时，Notebook 只展示文件级结果。

## 功能连接：MIC、MIM 与去偏平方 wPLI

连接估计在同一文件/记录节点/给药时点内跨多个有效 epoch 完成。MIC/MIM 使用四通道脑区集合的多变量估计；wPLI 保留全部跨脑区通道对。当前样例没有可用于动物层推断的身份信息。

In [3]:
if 'REAL' in run_mode:
    connectivity_rank = pd.read_csv(real_output / 'connectivity_rank_summary.csv')
    connectivity_checks = pd.read_csv(real_output / 'connectivity_input_checks.csv')
    connectivity_bands = pd.read_csv(real_output / 'connectivity_band_summary.csv')
    connectivity_spectrum = pd.read_csv(real_output / 'connectivity_spectrum.csv')
    print('Connectivity status:', manifest['connectivity_status'])
    print('Epochs:', manifest['n_epochs'], '| effective duration (s):', manifest['effective_valid_duration_s'])
    print('Spectrum rows:', len(connectivity_spectrum), '| band rows:', len(connectivity_bands))
    display(connectivity_rank[['region', 'n_channels', 'numerical_rank', 'variance_rank', 'selected_rank', 'covariance_condition_number']])
    display(connectivity_checks)
    display(connectivity_bands[['method', 'region_a', 'region_b', 'band', 'value_raw_or_summary', 'value_strength', 'n_epochs', 'rank_seed', 'rank_target', 'status']].head(24))
else:
    print('Connectivity is not run in synthetic fallback mode.')
    display(manifest)

Connectivity status: ok
Epochs: 21 | effective duration (s): 105.0
Spectrum rows: 53028 | band rows: 108


,region,n_channels,numerical_rank,variance_rank,selected_rank,covariance_condition_number
0,M1,4,4,3,3,361.065135
1,STR,4,4,3,3,230.673780
2,PF,4,4,3,3,141.365028
3,SNr,4,4,2,2,178.325602


,check,value,status,note
0,valid_epoch_count,21,ok,quality fail and nonfinite epochs excluded; wa...
1,effective_valid_duration_s,105.0,ok,sum of retained epoch durations; epochs are no...
2,quality_fail_epoch_count,0,ok,quality flags are retained in the audit tables
3,quality_warn_epoch_count,1,warn,warn epochs are retained unless quality status...
4,nonfinite_epoch_count,0,ok,nonfinite epochs are excluded from spectral es...
5,low_frequency_edge,2.0,ok,connectivity fmin=2 Hz; known/configured high-...
6,high_frequency_edge,100.0,ok,connectivity fmax=100 Hz; known/configured low...
7,line_noise_policy,"50.0,60.0",ok,line-noise bins are flagged/excluded in band s...
8,reference_policy,acquisition_reference_preserved,ok,"no bipolar, within-region average, orthogonali..."


,method,region_a,region_b,band,value_raw_or_summary,value_strength,n_epochs,rank_seed,rank_target,status
0,mic,M1,PF,delta,0.559186,0.559186,21,3,3,ok
1,mic,M1,PF,theta,0.374513,0.374513,21,3,3,ok
2,mic,M1,PF,alpha,0.202381,0.202381,21,3,3,ok
3,mic,M1,PF,beta,0.162689,0.162689,21,3,3,ok
4,mic,M1,PF,low_gamma,0.165765,0.165765,21,3,3,ok
5,mic,M1,PF,high_gamma,0.093092,0.093092,21,3,3,ok
6,mic,M1,SNr,delta,0.263513,0.263513,21,3,2,ok
7,mic,M1,SNr,theta,0.304391,0.304391,21,3,2,ok
8,mic,M1,SNr,alpha,0.252837,0.252837,21,3,2,ok
9,mic,M1,SNr,beta,0.153423,0.153423,21,3,2,ok


### 连接结果的当前限制

本次只支持单文件描述性连接结果；等量条件抽样在单文件上不适用，Granger/时间反转校正默认关闭。MIM 保留未归一化原值，wPLI 不开平方且不把负估计截为零。不要把通道对、epoch 或保留维度当成动物样本。